In [0]:
# Read the values passed from the job UI FOR DEPLOYMENT
import os
import json

import json

# Define the widgets
dbutils.widgets.text("status", "")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("data_ingestion_volume", "")
dbutils.widgets.text("run_time", "")
dbutils.widgets.text("sales_data", "")
dbutils.widgets.text("delta_table_volume", "")
dbutils.widgets.text("delta_table_schema_store", "")
dbutils.widgets.text("delta_table_checkpoints", "")
dbutils.widgets.text("delta_table_path", "")
dbutils.widgets.text("sales_table", "")
dbutils.widgets.text("sales_delta_table_schema_path", "")
dbutils.widgets.text("sales_delta_table_checkpoint_path", "")
dbutils.widgets.text("sales_delta_table_location", "")

# debugging for notbook parameters not being able to recieve from job-1 notebook
kvs = {k: dbutils.widgets.get(k) for k in ["status","catalog","schema","data_ingestion_volume","run_time","sales_data"]}
print("Widget values:", kvs)

# Databricks automatically creates widgets from notebook_params since I sent notebook_parmas from the upstream job I just have to get the params using dbutils
if kvs:
    # use the widgets
    status = dbutils.widgets.get("status")
    catalog = dbutils.widgets.get("catalog")
    schema = dbutils.widgets.get("schema")
    data_ingestion_volume = dbutils.widgets.get("data_ingestion_volume")

    print("Received params from Job 1: Running in Job mode")
    print(status, catalog, schema, data_ingestion_volume)

    sales_data = dbutils.widgets.get("sales_data")

    # catalog = dbutils.widgets.get("catalog")
    # schema  = dbutils.widgets.get("schema")
    delta_table_volume = dbutils.widgets.get("delta_table_volume")
    # data_ingestion_volume = dbutils.widgets.get("data_ingestion_volume")

    # delta table related paths
    delta_table_schema_store = dbutils.widgets.get("delta_table_schema_store")
    delta_table_checkpoints = dbutils.widgets.get("delta_table_checkpoints")
    delta_table_path = dbutils.widgets.get("delta_table_path")

    # data ingestion raw file related paths
    # orders_data = dbutils.widgets.get("orders_data")

    # customer_table_name
    sales_table = dbutils.widgets.get("sales_table")

    # for deployment
    sales_delta_table_schema_path = dbutils.widgets.get("sales_delta_table_schema_path")
    sales_delta_table_checkpoint_path = dbutils.widgets.get("sales_delta_table_checkpoint_path")
    sales_delta_table_location = dbutils.widgets.get("sales_delta_table_location")

    # databricks Job orchestration API
    third_job_id = dbutils.widgets.get("third_job_id")
    databricks_host = dbutils.widgets.get("databricks_host")
    job_orchestration_token = dbutils.widgets.get("job_orchestration_token")
else:
    print("No notebook_params received. Running in interactive mode.")
    # fallback defaults for testing
    status = "testing"
    catalog = "job_orchestration"
    schema = "default"
    volume = "job_orchestration_volume"
    run_time = "N/A"
    
    catalog = "job_orchestration"
    schema  = "default"
    delta_table_volume = "sales_volume"
    data_ingestion_volume = "job_orchestration_volume"

    # data table related paths
    delta_table_schema_store = "/Volumes/job_orchestration/default/sales_volume/schema_store"
    delta_table_checkpoints = "/Volumes/job_orchestration/default/sales_volume/checkpoints"
    delta_table_path = "/Volumes/job_orchestration/default/sales_volume/sales_table"

    # data ingestion related raw files related paths
    # customers_data = "/Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/customers_data/"
    # orders_data = "/Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/orders_data/"
    sales_data = "/Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/sales_data/"

    # customer_table_name
    sales_table="sales_table"

    # for testing
    sales_delta_table_schema_path          = "/Volumes/job_orchestration/default/sales_volume/schema_store/"
    sales_delta_table_checkpoint_path      = "/Volumes/job_orchestration/default/sales_volume/checkpoints/"
    # If I pass this customer_delta_table path like this then the table created will not be registered in the unity catalog and as a result it will not be accesible by using sql commands
    # customer_delta_table_location       = "/Volumes/job_orchestration/default/sales_volume/sales_table/"
    # If i pass this customer_delta_table path like this then the table created will be registered in the unity catalog and as a result it will be accesible by suing sql commands
    sales_delta_table_location       = "job_orchestration.default.sales_table"

print("Catalog:", catalog)
print("Schema:", schema)

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{delta_table_volume};")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{data_ingestion_volume};")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{sales_table} USING DELTA;
""")

In [0]:
from pyspark.sql.utils import AnalysisException

# Check if csv file in customer_data location exists or not
def path_has_files(path):
    try:
        files = dbutils.fs.ls(path)
        # Filter out only real data files (csv, parquet, etc.)
        data_files = [f for f in files if f.name.lower().endswith(".csv")]
        return len(data_files) > 0
    except Exception:
        # Directory does NOT exist
        return False
data_ingested = False
if path_has_files(sales_data):
    print("✔ CSV files found — starting Auto Loader ingestion...")
    df = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", sales_delta_table_schema_path)
            .load(sales_data)
    )

    # This code generates the table that is registered in the unity catalog hence it can be accessed via sql commands 
    (
        df.writeStream
        .option("checkpointLocation", sales_delta_table_checkpoint_path)
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        # .table("job_orchestration.default.customers_table")
        .table(sales_delta_table_location)
    )
    data_ingested = True
else:
    print("✘ No CSV files found — skipping ingestion.")
    data_ingested = False
    pass


In [0]:
import json
from pyspark.sql import Row
import requests

# Now trigger Ingest_sales_data using databricks api from this task in job 1 and send the relavant required data to job 2 which is essential to run data-ingestion logic in job 2
if data_ingested:
    resp = requests.post(
        databricks_host+"/api/2.1/jobs/run-now",
        headers={"Authorization": f"Bearer {job_orchestration_token}"},
        json={"job_id": third_job_id, 
            "task_key":"join_customer_and_sales_data", # task_key is required for multi-task jobs
            "notebook_params": {
                    "status": "success",
                    "catalog": catalog,
                    "schema": schema,
                    "data_ingestion_volume": data_ingestion_volume,
                    "databricks_host":databricks_host,
                    "job_orchestration_token":job_orchestration_token
                }},
    )

    # Debugging
    # since job orchestration is not working properly, I am inspecting the response I get from this api call that I made to trigger job-2's execution programmatically.
    print(resp.status_code)
    print(resp.text)        # should include run_id or error details
    # If status 200, parse JSON:
    print(resp.json())
else:
    print(f"No data ingested triggering job : {third_job_id} skipped")